In [1]:
# Cài đủ thư viện cần cho Streamlit
!pip install streamlit pyngrok gensim bertopic sentence-transformers \
             umap-learn hdbscan plotly nltk wordcloud pyLDAvis -q

import nltk
for pkg in ['punkt','punkt_tab','stopwords','wordnet']:
    nltk.download(pkg, quiet=True)

print('✅ Thư viện sẵn sàng')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 77.7 MB/s eta 0:00:00
✅ Thư viện sẵn sàng


In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Kiểm tra Drive đã mount thành công
import os
if os.path.exists('/content/drive/MyDrive'):
    print(' Drive đã mount tại /content/drive/MyDrive')
else:
    print(' Drive chưa mount — chạy lại cell này')

Mounted at /content/drive
 Drive đã mount tại /content/drive/MyDrive


In [3]:

DRIVE_PATH = '/content/drive/MyDrive/ĐACN3/streamlit_package'

import os, shutil

if not os.path.exists(DRIVE_PATH):
    print(f' Không tìm thấy: {DRIVE_PATH}')
    print('\n Kiểm tra lại đường dẫn. Cách tìm:')
    print('   1. Mở drive.google.com')
    print('   2. Tìm folder streamlit_package')
    print('   3. Click chuột phải → "View location"')
    print('   4. Sửa biến DRIVE_PATH cho khớp')
else:
    print(f' Tìm thấy folder Drive: {DRIVE_PATH}')
    print('   Đang copy file về /content/...')

    # Copy tất cả file từ folder Drive về /content (Streamlit cần file ở /content)
    for fname in os.listdir(DRIVE_PATH):
        src = os.path.join(DRIVE_PATH, fname)
        dst = os.path.join('/content', fname)
        if os.path.isfile(src):
            shutil.copy(src, dst)
            print(f'    {fname}')

    print('\n🎉 Copy xong! Sẵn sàng chạy Streamlit.')

 Tìm thấy folder Drive: /content/drive/MyDrive/ĐACN3/streamlit_package
   Đang copy file về /content/...
    amazon_reviews_with_topics.csv
    lda_model.id2word
    lda_dictionary.dict
    bert_topic_words.pkl
    lda_model.state
    intrinsic_metrics.csv
    trigram.pkl
    bert_results.csv
    lda_model.expElogbeta.npy
    bigram.pkl
    lda_model
    topic_names.pkl
    app.py
    eval_metrics.json

🎉 Copy xong! Sẵn sàng chạy Streamlit.


In [4]:
import os

REQUIRED = [
    'app.py',
    'lda_model', 'lda_model.expElogbeta.npy', 'lda_model.id2word', 'lda_model.state',
    'lda_dictionary.dict',
    'topic_names.pkl', 'bigram.pkl', 'trigram.pkl', 'bert_topic_words.pkl',
    'amazon_reviews_with_topics.csv',
]
OPTIONAL = ['bert_results.csv', 'eval_metrics.json', 'intrinsic_metrics.csv']

print(' File BẮT BUỘC (cho Streamlit chạy được):')
missing = []
for f in REQUIRED:
    path = f'/content/{f}'
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f'   {f:35s} ({size:,} bytes)')
    else:
        print(f'   {f:35s} — THIẾU!')
        missing.append(f)

print('\n File TÙY CHỌN (cho Tab 4 & 5):')
for f in OPTIONAL:
    path = f'/content/{f}'
    if os.path.exists(path):
        print(f'   {f}')
    else:
        print(f'    {f} — Không có (Tab tương ứng sẽ báo cảnh báo)')

print()
if missing:
    print(f' Thiếu {len(missing)} file BẮT BUỘC — không chạy được Streamlit')
    print('   Kiểm tra lại folder streamlit_package trên Drive')
else:
    print(' Đủ file BẮT BUỘC — sẵn sàng chạy Streamlit ở cell tiếp theo!')

 File BẮT BUỘC (cho Streamlit chạy được):
   app.py                              (26,160 bytes)
   lda_model                           (41,590 bytes)
   lda_model.expElogbeta.npy           (245,380 bytes)
   lda_model.id2word                   (285,828 bytes)
   lda_model.state                     (280,942 bytes)
   lda_dictionary.dict                 (286,076 bytes)
   topic_names.pkl                     (257 bytes)
   bigram.pkl                          (55,868 bytes)
   trigram.pkl                         (106,272 bytes)
   bert_topic_words.pkl                (2,981 bytes)
   amazon_reviews_with_topics.csv      (41,490,017 bytes)

 File TÙY CHỌN (cho Tab 4 & 5):
   bert_results.csv
   eval_metrics.json
   intrinsic_metrics.csv

 Đủ file BẮT BUỘC — sẵn sàng chạy Streamlit ở cell tiếp theo!


In [5]:
# Điền token ngrok của bạn
NGROK_TOKEN = '3Da8C37r0maopLgSfxZMaDpbrXv_wYRgiWLTYfFmn2FpdWWJ'  # ← Điền token vào đây

if not NGROK_TOKEN:
    print('⚠️  Vui lòng điền NGROK_TOKEN ở dòng trên rồi chạy lại cell này')
    print('   Đăng ký miễn phí: https://dashboard.ngrok.com')
else:
    from pyngrok import ngrok
    import subprocess, time, requests

    # ════════════════════════════════════════════════════════════
    # BƯỚC 1: Force kill tất cả tunnel cũ qua ngrok API
    # ════════════════════════════════════════════════════════════
    print('🔪 Đang dọn dẹp tunnel cũ...')
    try:
        ngrok.kill()
        time.sleep(2)
    except: pass

    # Kill tunnel online qua ngrok API
    try:
        headers = {
            'Authorization': f'Bearer {NGROK_TOKEN}',
            'Ngrok-Version': '2'
        }
        # Endpoints (mới)
        r = requests.get('https://api.ngrok.com/endpoints', headers=headers)
        if r.status_code == 200:
            for e in r.json().get('endpoints', []):
                eid = e.get('id')
                if eid:
                    requests.delete(f'https://api.ngrok.com/endpoints/{eid}', headers=headers)
                    print(f'   ❌ Dừng endpoint cũ: {e.get("public_url","?")}')
        # Tunnels (legacy)
        r2 = requests.get('https://api.ngrok.com/tunnels', headers=headers)
        if r2.status_code == 200:
            for t in r2.json().get('tunnels', []):
                tid = t.get('id')
                if tid:
                    requests.delete(f'https://api.ngrok.com/tunnels/{tid}', headers=headers)
                    print(f'   ❌ Dừng tunnel cũ: {t.get("public_url","?")}')
        time.sleep(3)
    except Exception as e:
        print(f'   ⚠️  Không kill được qua API: {e}')

    print('✅ Đã dọn xong\n')

    # ════════════════════════════════════════════════════════════
    # BƯỚC 2: Kill process streamlit cũ
    # ════════════════════════════════════════════════════════════
    subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
    time.sleep(2)

    # ════════════════════════════════════════════════════════════
    # BƯỚC 3: Khởi chạy Streamlit + tạo tunnel mới
    # ════════════════════════════════════════════════════════════
    ngrok.set_auth_token(NGROK_TOKEN)

    proc = subprocess.Popen(
        ['streamlit', 'run', '/content/app.py',
         '--server.port', '8501',
         '--server.headless', 'true'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    print('⏳ Đợi Streamlit khởi động...')
    time.sleep(8)

    try:
        public_url = ngrok.connect(8501)
        print('='*60)
        print(f'🌐 Streamlit demo: {public_url}')
        print('='*60)
        print('→ Mở link trên để xem dashboard 6 tab')
        print('→ Lần đầu mở có thể chậm 10-15s (load model)')
    except Exception as e:
        print(f'❌ Lỗi tạo tunnel: {e}')
        print('\n💡 Fix thủ công:')
        print('   1. Mở https://dashboard.ngrok.com/agents')
        print('   2. Stop tất cả tunnel đang online')
        print('   3. Chạy lại cell này')

🔪 Đang dọn dẹp tunnel cũ...
✅ Đã dọn xong

⏳ Đợi Streamlit khởi động...
🌐 Streamlit demo: NgrokTunnel: "https://uneven-catty-uninsured.ngrok-free.dev" -> "http://localhost:8501"
→ Mở link trên để xem dashboard 6 tab
→ Lần đầu mở có thể chậm 10-15s (load model)
